In [1]:
import sys
import torch

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    raise RuntimeError(
        "GPU is not available. Enable a GPU runtime in Colab."
    )

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [2]:
%pip install -q "transformers==4.51.3" "accelerate==1.6.0" "pandas==2.2.3"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
from google.colab import files

uploaded = files.upload()

Saving test.csv to test.csv


In [4]:
import pandas as pd

TEST_DATA_PATH = "/content/test.csv"

if "test.csv" not in uploaded:
    raise FileNotFoundError("Please upload the original test.csv file.")

test_df = pd.read_csv(TEST_DATA_PATH)

required_columns = {"instruction", "category", "intent"}

missing_columns = required_columns - set(test_df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

if len(test_df) != 2428:
    raise ValueError(
        f"Expected 2428 test examples, received {len(test_df)}."
    )

if test_df[list(required_columns)].isnull().any().any():
    raise ValueError("Test dataset contains missing required values.")

print("Dataset loaded successfully.")
print("Number of examples:", len(test_df))
print("Columns:", test_df.columns.tolist())
print("Categories:", test_df["category"].nunique())
print("Intents:", test_df["intent"].nunique())

display(test_df.head())

Dataset loaded successfully.
Number of examples: 2428
Columns: ['flags', 'instruction', 'category', 'intent', 'response', 'normalized_instruction']
Categories: 11
Intents: 27


,flags,instruction,category,intent,response,normalized_instruction
0,BL,help me to see how long it takes for the artic...,DELIVERY,delivery_period,We understand your eagerness to know the estim...,help me to see how long it takes for the artic...
1,BILM,can you give me information about opening {{Ac...,ACCOUNT,create_account,Of course! I'd be delighted to provide you wit...,can you give me information about opening {{ac...
2,BCIMQ,i have got to buy some items how can i do it,ORDER,place_order,Thank you for reaching out! It's great to see ...,i have got to buy some items how can i do it
3,BLMN,I can't notify of problems with a registration,ACCOUNT,registration_problems,We understand that you're experiencing some di...,i can't notify of problems with a registration
4,B,I don't know what I have to do to create a {{A...,ACCOUNT,create_account,I'm delighted that you're interested in creati...,i don't know what i have to do to create a {{a...


**Load the Original Qwen model**

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this inference run.")

# Select a GPU-compatible numerical precision.
if torch.cuda.is_bf16_supported():
    MODEL_DTYPE = torch.bfloat16
else:
    MODEL_DTYPE = torch.float16

print("Loading tokenizer:", MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Tokenizer class:", tokenizer.__class__.__name__)
print("Native chat template available:", tokenizer.chat_template is not None)
print("EOS token:", tokenizer.eos_token)

if tokenizer.chat_template is None:
    raise RuntimeError("The model tokenizer does not contain a chat template.")

# Decoder-only models should use left padding for batched generation.
tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("\nLoading original model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=MODEL_DTYPE,
    device_map="auto",
)

model.eval()

print("\nModel loaded successfully.")
print("Model device:", model.device)
print("Model dtype:", MODEL_DTYPE)

Loading tokenizer: Qwen/Qwen2.5-0.5B-Instruct


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (2 outputs left out)

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model loaded successfully.
Model device: cuda:0
Model dtype: torch.bfloat16


In [6]:
SYSTEM_PROMPT = (
    "Classify customer support requests. "
    'Return only a JSON object with keys "category" and "intent".'
)


def build_inference_prompt(instruction):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": str(instruction),
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


test_prompts = [
    build_inference_prompt(instruction)
    for instruction in test_df["instruction"]
]

print("Total prompts:", len(test_prompts))
print("\nExample inference prompt:\n")
print(test_prompts[0])

Total prompts: 2428

Example inference prompt:

<|im_start|>system
Classify customer support requests. Return only a JSON object with keys "category" and "intent".<|im_end|>
<|im_start|>user
help me to see how long it takes for the article to arrive<|im_end|>
<|im_start|>assistant



In [7]:
import torch
from tqdm.auto import tqdm

BATCH_SIZE = 8
MAX_NEW_TOKENS = 64

all_outputs = []

model.eval()

for start_index in tqdm(
    range(0, len(test_prompts), BATCH_SIZE),
    desc="Baseline inference",
):
    batch_prompts = test_prompts[
        start_index : start_index + BATCH_SIZE
    ]

    inputs = tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Remove the original padded prompt tokens.
    generated_only_ids = generated_ids[:, inputs["input_ids"].shape[1]:]

    batch_outputs = tokenizer.batch_decode(
        generated_only_ids,
        skip_special_tokens=True,
    )

    all_outputs.extend(
        output.strip() for output in batch_outputs
    )

assert len(all_outputs) == len(test_df)

print("Inference completed.")
print("Generated predictions:", len(all_outputs))

Baseline inference:   0%|          | 0/304 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Inference completed.
Generated predictions: 2428


In [8]:
from pathlib import Path

baseline_df = test_df[
    ["instruction", "category", "intent"]
].copy()

baseline_df["raw_output"] = all_outputs

OUTPUT_DIR = Path("/content/baseline_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "baseline_predictions.csv"
OUTPUT_JSON = OUTPUT_DIR / "baseline_predictions.jsonl"

baseline_df.to_csv(OUTPUT_CSV, index=False)
baseline_df.to_json(
    OUTPUT_JSON,
    orient="records",
    lines=True,
    force_ascii=False,
)

print("Saved:", OUTPUT_CSV)
print("Saved:", OUTPUT_JSON)

print("\nSample predictions:")
display(baseline_df.head(10))

Saved: /content/baseline_results/baseline_predictions.csv
Saved: /content/baseline_results/baseline_predictions.jsonl

Sample predictions:


,instruction,category,intent,raw_output
0,help me to see how long it takes for the artic...,DELIVERY,delivery_period,"```json\n{\n ""category"": ""Customer Support"",\..."
1,can you give me information about opening {{Ac...,ACCOUNT,create_account,"{\n ""category"": ""Customer Support"",\n ""inten..."
2,i have got to buy some items how can i do it,ORDER,place_order,"```json\n{\n ""category"": ""Request for Informa..."
3,I can't notify of problems with a registration,ACCOUNT,registration_problems,"```json\n{\n ""category"": ""User Support"",\n ""..."
4,I don't know what I have to do to create a {{A...,ACCOUNT,create_account,"```json\n{\n ""category"": ""Account Creation"",\..."
5,where do i check the cancellation penalty,CANCEL,check_cancellation_fee,"```json\n{\n ""category"": ""Help"",\n ""intent"":..."
6,assistance to check hw soon can i expect the o...,DELIVERY,delivery_period,"```json\n{\n ""category"": ""Request for Informa..."
7,I need help to talk to a person,CONTACT,contact_human_agent,"```json\n{\n ""category"": ""customer service"",\..."
8,I don't know how to correct the data included ...,ACCOUNT,edit_account,"```json\n{\n ""category"": ""Help Request"",\n ""..."
9,want help to cancel order {{Order Number}},ORDER,cancel_order,"```json\n{\n ""category"": ""Customer Support"",\..."


In [9]:
from google.colab import files

files.download(str(OUTPUT_CSV))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>